# Notebook referente ao exercício 5.2 - Modelagem de sinal de radar

Utilizando o modelo de equação de Claerbout de grande abertura, o campo elétrico $u(x, z) = E_y(x, z)$ na atmosfera sobre a superfície do mar, considerando o domínio de propagação $\Omega = \{(x, z) \mid z \in [0, H], x \in [0, X]\}$, onde:

* $z = H = 200$ m é uma fronteira transparente.
* $z = 0$ é a superfície do mar, na qual a condição de Dirichlet $u = 0$ é imposta.

Satisfaz a seguinte equação:

$$u_{xx} + u_{zz} + k^2(1 + n)u = -\delta(x)Q(z)$$

Onde:
* $k = 2\pi f / c$ (número de onda).
* $f$ é a frequência.
* $c$ é a velocidade da luz no vácuo.
* $n(x, z)$ é o quadrado do índice de refração.
* $Q(z)$ é a diretividade da fonte (uma antena parabólica).

A função $Q(z)$ define a condição inicial $u(0, z) = u_0(z) = Q(z)$:

$$Q(z) = A \frac{k\beta}{2\sqrt{\pi} \log_{10} 2} \exp(-ik\theta_0 z) \exp \left( -\frac{\beta^2}{8 \log_{10} 2} k^2(z - z_0)^2 \right)$$

Com os seguintes parâmetros
* $\theta_0$: ângulo de elevação do feixe.
* $z_0 = 10$ m: altitude da fonte.
* $\beta = 35^\circ$: largura do feixe.

---
As configurações da simulação são as seguintes:
* **Frequência:** $f = 300$ MHz (adotada para acelerar os cálculos).
* **Alcance:** Realizar cálculos para até $X = 20$ km.
* **Ângulos de Teste:** $\theta_0 = 5^\circ, 10^\circ, 20^\circ$.

**Perfil do Índice de Refração $n(z)$:**

$$n(z) = 10^{-5} \times \begin{cases} 0, & \text{if } z \geq 100 \text{ m} \\ -1 + z/100, & \text{if } z \in [0, 100 \text{ m}] \end{cases}$$


Para resolver o problema, vamos considerar apenas a propagação para a direita, de modo que a nossa equação diferencial fique da seguinte maneira,

$$\frac{\partial u}{\partial x}=i\sqrt{k^2(1+n(x,z))+\frac{\partial^2u}{\partial z^2}}$$

e consideramos a condição 

$$u(0,z)=Q(z).$$

Podemos usar a aproximação de Padé(1,1), para o lado direito da nossa equação diferencial, nos deixanco com,

$$\frac{\partial u}{\partial x}=ik\frac{1+\frac{1}{4}(n(x,z)+\frac{1}{k^2}\frac{\partial^2}{\partial z^2})}{1-\frac{1}{4}(n(x,z)+\frac{1}{k^2}\frac{\partial^2}{\partial z^2})}u$$

Para simplificar as contas, vamos considerar
$$\hat{A}=(n(x,z)+\frac{1}{k^2}\frac{\partial^2}{\partial z^2})$$

Partindo dessa equação diferencial, podemos discretizar a malha que contém a nossa função, tomando $u_x=\frac{u_{n+1,j}-u_{n,j}}{\Delta x}$ e $u=\frac{u_{n+1,j}+u_{n,j}}{2}$. Com isso, obtemos

$$[2-\hat{A}/2-i\Delta xk(1+\hat{A}/4)]u_{n+1,j}=[2-\hat{A}/2+i\Delta xk(1+\hat{A}/4)]u_{n,j}$$

Dessa maneira, construímos as matrizes $B$ e $C$ como

$$B=(1-\frac{i\Delta xk}{2})-\frac{1}{4}(1+\frac{i\Delta xk}{2})\hat{X}$$
$$C=(1+\frac{i\Delta xk}{2})-\frac{1}{4}(1-\frac{i\Delta xk}{2})\hat{X}$$

Definimos essas matrizes considerando que a função $n$, que aparece em $\hat{A}$, não depende de $x$, ou seja, $n(z)$ é apenas função de $z$.

Determinamos agora, na malha referente a $z$, a segunda derivada em relação a $z$ que aparece em $\hat{A}$,

$$\frac{\partial^2 u_j}{\partial z^2}=\frac{u_{j+1}-2u_{j}+u_{j-1}}{\Delta z^2}$$

Definindo uma nova variável $\gamma=\frac{1}{k^2\Delta z^2}$, obtemos, que a aplicação de $\hat{A}$ em $u_j$, nos dá o seguinte,

$$\hat{A}u_j=[(n_j-2\gamma)u_j+\gamma (u_{j+1}+u_{j-1})]$$

Desse modo, podemos concluir que $\hat{A}$ é uma matriz tridiagonal, com elemenos da diagonal sendo $n_j-2\gamma$, e diagonais inferior e superior com elementos $\gamma$. Com isso, vemos que a matriz que chegamos $B$ também será tridiagonal, de modo que,

* Diagonal principal:
$$(1-\frac{i\Delta xk}{2})-\frac{1}{4}(1+\frac{i\Delta xk}{2})(n_j-2\gamma)$$
* Diagonais inferiores e superiores:
$$-\frac{1}{4}(1+\frac{i\Delta xk}{2})\gamma$$

E para a matriz $C$, temos 

* Diagonal principal:
$$(1+\frac{i\Delta xk}{2})-\frac{1}{4}(1-\frac{i\Delta xk}{2})(n_j-2\gamma)$$
* Diagonais inferiores e superiores:
$$-\frac{1}{4}(1-\frac{i\Delta xk}{2})\gamma$$

Válidas para os pontos internos $j$, mas não para o bordo. Com a condição de contorno de Dirichlet para $z=0$ para o bordo inferior. Para o bordo superior, podemos extender o domínio em um valor $\delta$ e utilizar uma função $\sigma(z)$ de amortecimento.

